In [1]:
!pip install -q x-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.4 MB/s eta 0:00:00


In [2]:
import importlib, sys

spec = importlib.util.find_spec("x_transformers")
print("x_transformers installed?", spec is not None)

if spec is None:
    raise RuntimeError(
        "x_transformers ยังไม่ถูกติดตั้ง (มักเกิดจากเน็ต/DNS ของ Kaggle). "
        "ลอง restart session แล้วติดตั้งใหม่ หรือใช้ torch.nn.TransformerEncoder แทน"
    )


x_transformers installed? True


In [3]:
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score, classification_report, accuracy_score

from transformers import AutoTokenizer, AutoModel
from x_transformers import TransformerWrapper, Decoder

In [4]:
TRAIN_CSV = "/kaggle/input/datasets/ratthachat/pythainlp-prachatai-67k/prachatai_train.csv"
VAL_CSV   = "/kaggle/input/datasets/ratthachat/pythainlp-prachatai-67k/prachatai_validation.csv"
TEST_CSV  = "/kaggle/input/datasets/ratthachat/pythainlp-prachatai-67k/prachatai_test.csv"

MODEL_SAVE_PATH = "wangchan_xtrans_freeze.pt"
MODEL_RECORD_PATH = "/kaggle/input/models/nareupol/step51/pytorch/default/1/wangchan_xtrans_freeze.pt"

MAX_LEN = 256
BATCH_SIZE = 128
EPOCHS = 100
# PATIENCE = 8
LR = 2e-4

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [6]:
if os.path.exists(MODEL_RECORD_PATH):
    print("Debugging checkpoint...")
    ckpt = torch.load(MODEL_RECORD_PATH, map_location=device)
    start_epoch = ckpt["epoch"] + 1
    print("Resume from epoch(Debugging)", start_epoch)

Debugging checkpoint...
Resume from epoch(Debugging) 50


In [7]:
LABEL_COLS = [
    "politics", "human_rights", "quality_of_life", "international",
    "social", "environment", "economics", "culture", "labor",
    "national_security", "ict", "education"
]

In [8]:
hf_tokenizer = AutoTokenizer.from_pretrained(
    "airesearch/wangchanberta-base-att-spm-uncased",
    use_fast=True
)
PAD_ID = hf_tokenizer.pad_token_id

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/905k [00:00<?, ?B/s]

In [9]:
def tokenize_texts(texts):
    """
    PyThaiNLP -> WangchanBERTa subword
    """
    enc = hf_tokenizer(
        texts,
        is_split_into_words=False,
        truncation=True,
        max_length=MAX_LEN,
        padding=False, # True if not use collate_fn
        # return_tensors="pt"
    )
    # print(enc["input_ids"][0])
    return enc["input_ids"]

def load_dataset(csv_path):
    df = pd.read_csv(csv_path)
    # texts = df["body_text"].head(100).astype(str).tolist()
    # labels = df[LABEL_COLS].head(100).values.astype(np.float32)
    texts = df["body_text"].astype(str).tolist()
    labels = df[LABEL_COLS].values.astype(np.float32)
    print("path:", csv_path, "\t shape:",df.shape)

    X = tokenize_texts(texts)
    return X, labels

X_train, y_train = load_dataset(TRAIN_CSV)
X_val, y_val     = load_dataset(VAL_CSV)
X_test, y_test   = load_dataset(TEST_CSV)


path: /kaggle/input/datasets/ratthachat/pythainlp-prachatai-67k/prachatai_train.csv 	 shape: (54379, 17)
path: /kaggle/input/datasets/ratthachat/pythainlp-prachatai-67k/prachatai_validation.csv 	 shape: (6721, 17)
path: /kaggle/input/datasets/ratthachat/pythainlp-prachatai-67k/prachatai_test.csv 	 shape: (6789, 17)


In [10]:
class ThaiDataset(Dataset):
    def __init__(self, X, y):
        # self.X = torch.as_tensor(X, dtype=torch.long)
        self.X = [torch.tensor(seq, dtype=torch.long) for seq in X]
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def collate_fn(batch):
    seqs, labels = zip(*batch)

    padded = pad_sequence(
        seqs, batch_first=True, padding_value=PAD_ID
    )
    attn_mask = (padded != PAD_ID).long()
    labels = torch.stack(labels)
    return (
        padded,
        attn_mask,
        labels
    )

train_loader = DataLoader(
    ThaiDataset(X_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    ThaiDataset(X_val, y_val),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    ThaiDataset(X_test, y_test),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [11]:
encoder = AutoModel.from_pretrained(
    "airesearch/wangchanberta-base-att-spm-uncased"
)

for p in encoder.parameters():
    p.requires_grad = False

model.safetensors:   0%|          | 0.00/423M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CamembertModel LOAD REPORT from: airesearch/wangchanberta-base-att-spm-uncased
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
class WangchanXTClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = encoder
        self.hidden = encoder.config.hidden_size  # 768

        self.decoder = TransformerWrapper(
            num_tokens=1,  # dummy
            max_seq_len=MAX_LEN,
            attn_layers=Decoder(
                dim=self.hidden,
                depth=4,
                heads=4,
                cross_attend=True,
                # causal=False
            )
        )

        self.classifier = nn.Linear(self.hidden, len(LABEL_COLS))

    def forward(self, input_ids, attention_mask):
        # ---- Encoder ----
        enc = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state

        # ---- Decoder ----
        
        dummy_ids = input_ids.new_zeros(input_ids.shape)

        dec = self.decoder(
            x=dummy_ids,
            context=enc,
            context_mask=attention_mask.bool(),
            return_embeddings=True
        )

        mask = attention_mask.unsqueeze(-1)
        pooled = (dec * mask).sum(1) / mask.sum(1).clamp(min=1)

        return self.classifier(pooled)

In [13]:
 # pos_weight สำหรับ imbalance (กัน label ที่หายาก) 
pos = y_train.sum(axis=0)
neg = y_train.shape[0] - pos
pos_weight = torch.tensor(neg / (pos + 1e-8), dtype=torch.float32).to(device)

In [14]:
model = WangchanXTClassifier().to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR
)

In [15]:
class Save_model:
    def __init__(self):
        self.best = 0.0
        # self.counter = 0
        # self.patience = patience

    def step(self, score, model, epoch):
        if score > self.best:
            self.best = score
            # self.counter = 0
            torch.save({
                "epoch": epoch,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "best": self.best
            }, MODEL_SAVE_PATH)
            return False
        elif epoch == EPOCHS - 1:  
            print("save final model successfully")
            torch.save({
                "epoch": epoch,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "best": score
            }, MODEL_SAVE_PATH)
            return False

In [16]:

start_epoch = 0

save_model = Save_model()

if os.path.exists(MODEL_RECORD_PATH):
    print("Loading checkpoint...")
    ckpt = torch.load(MODEL_RECORD_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_epoch = ckpt["epoch"] + 1
    save_model.best = ckpt["best"]
    print("Resume from epoch", start_epoch)

Loading checkpoint...
Resume from epoch 50


In [17]:

# ---------------- TRAIN ----------------
print("Training...")
for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0

    for Xb, maskb, yb in train_loader:
        Xb = Xb.to(device, non_blocking=True)
        maskb = maskb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        optimizer.zero_grad()

        preds = model(Xb, maskb)
        loss = criterion(preds, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # ---- VALIDATION ----
    model.eval()
    yt, yp = [], []

    with torch.no_grad():
        for Xb, maskb, yb in val_loader:
            Xb = Xb.to(device, non_blocking=True)
            maskb = maskb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            preds = (model(Xb, maskb) > 0).int() # sigmoid
            yt.append(yb.cpu().numpy())
            yp.append(preds.cpu().numpy())

    yt = np.vstack(yt)
    yp = np.vstack(yp)
    val_f1 = f1_score(yt, yp, average="macro")

    print(
        f"Epoch {epoch+1:03d} | "
        f"Loss {total_loss:.4f} | "
        f"Val F1 {val_f1:.4f}"
    )

    if save_model.step(val_f1, model, epoch):
        print("save weight now...")
        break


Training...
Epoch 051 | Loss 63.1875 | Val F1 0.6670
Epoch 052 | Loss 48.4195 | Val F1 0.6744
Epoch 053 | Loss 41.9796 | Val F1 0.6721
Epoch 054 | Loss 37.7686 | Val F1 0.6792
Epoch 055 | Loss 35.6988 | Val F1 0.6793
Epoch 056 | Loss 35.4765 | Val F1 0.6797
Epoch 057 | Loss 34.6392 | Val F1 0.6766
Epoch 058 | Loss 33.2187 | Val F1 0.6818
Epoch 059 | Loss 29.5327 | Val F1 0.6813
Epoch 060 | Loss 29.4669 | Val F1 0.6756
Epoch 061 | Loss 29.5371 | Val F1 0.6857
Epoch 062 | Loss 27.6134 | Val F1 0.6881
Epoch 063 | Loss 25.6226 | Val F1 0.6842
Epoch 064 | Loss 25.6907 | Val F1 0.6851
Epoch 065 | Loss 27.9395 | Val F1 0.6838
Epoch 066 | Loss 26.5902 | Val F1 0.6846
Epoch 067 | Loss 24.1543 | Val F1 0.6746
Epoch 068 | Loss 24.3965 | Val F1 0.6797
Epoch 069 | Loss 21.8723 | Val F1 0.6837
Epoch 070 | Loss 21.9716 | Val F1 0.6874
Epoch 071 | Loss 24.9298 | Val F1 0.6800
Epoch 072 | Loss 23.2084 | Val F1 0.6784
Epoch 073 | Loss 21.5793 | Val F1 0.6724
Epoch 074 | Loss 22.4393 | Val F1 0.6792
Epoc

In [18]:
print("Testing...")

model.eval()
yt, yp = [], []
with torch.no_grad(): 
    for Xb, maskb, yb in test_loader: 
        Xb = Xb.to(device, non_blocking=True) 
        maskb = maskb.to(device, non_blocking=True) 
        yb = yb.to(device, non_blocking=True) 
        preds =(model(Xb, maskb) >= 0).int() 
        yt.append(yb.cpu().numpy()) 
        yp.append(preds.cpu().numpy()) 
        
yt = np.vstack(yt) 
yp = np.vstack(yp) 
acc = (yt == yp).mean()
acc_score_lib = accuracy_score(yt, yp)
f1_macro = f1_score(yt, yp, average='macro') 
f1_micro = f1_score(yt, yp, average='micro') 
f1_samples = f1_score(yt, yp, average='samples')
print(f"F1 micro: {f1_micro:.4f} | acc-elementwise: {acc:.4f} | acc-score-lib: {acc_score_lib:.4f}| F1 macro: {f1_macro:.4f} | Samples F1: {f1_samples:.4f}")
print(classification_report(yt, yp, digits=4))

Testing...
F1 micro: 0.7252 | acc-elementwise: 0.9242 | acc-score-lib: 0.4915| F1 macro: 0.6661 | Samples F1: 0.7190
              precision    recall  f1-score   support

           0     0.8321    0.8550    0.8434      3842
           1     0.6931    0.6651    0.6788      1511
           2     0.6594    0.6717    0.6655      1127
           3     0.7459    0.8129    0.7780       834
           4     0.4407    0.5082    0.4720       789
           5     0.7536    0.8199    0.7854       772
           6     0.6039    0.6551    0.6285       519
           7     0.6384    0.4925    0.5560       398
           8     0.7669    0.8743    0.8171       350
           9     0.4894    0.6154    0.5452       338
          10     0.7003    0.7123    0.7063       292
          11     0.5333    0.5020    0.5172       255

   micro avg     0.7122    0.7386    0.7252     11027
   macro avg     0.6548    0.6820    0.6661     11027
weighted avg     0.7147    0.7386    0.7254     11027
 samples avg     

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

In [19]:
def predict(text, threshold=0.5):
    # words = word_tokenize(text, engine="newmm")
    enc = hf_tokenizer(
        text,
        is_split_into_words=False,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN
    )
    

    with torch.no_grad():
        probs = model(
            enc["input_ids"].to(device),
            enc["attention_mask"].to(device)
        )[0]
        probs = torch.sigmoid(probs)

    return sorted(
        [(LABEL_COLS[i], float(probs[i].item()))
         for i in range(len(LABEL_COLS)) if probs[i] >= threshold],
        key=lambda x: x[1],
        reverse=True
    )

print("\nPREDICT EXAMPLES")
print(predict("รัฐบาลไทยประฤกาศนโยบายด้านสิ่งแวดล้อมใหม่"))
print(predict("แรงงานเรียกร้องสิทธิ์การทำงาน"))


PREDICT EXAMPLES
[('international', 0.9989758729934692), ('politics', 0.8918808102607727), ('quality_of_life', 0.753074586391449), ('social', 0.7345413565635681)]
[('environment', 0.9982035160064697), ('international', 0.9381355047225952), ('social', 0.9366563558578491), ('quality_of_life', 0.5323094129562378)]
